In [1]:
!pip install -q google-genai

In [2]:
import json, time
from google.colab import userdata
from google import genai
from google.genai import types

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
gclient = genai.Client(api_key=GEMINI_API_KEY)
# JUDGE_MODEL = "gemini-3-flash-preview"
JUDGE_MODEL = "gemini-3.1-flash-lite"

In [3]:
FILE_PATH = 'PEP_AUGMENTED_150.json'

with open(FILE_PATH, encoding='utf-8') as f:
    dataset = json.load(f)

print(f"검증 대상: {len(dataset)}개")

검증 대상: 150개


In [4]:
JUDGE_DEFINITIONS = """[평가기준 정의]
- 완전성(COMP): RFP/사용자 요구가 계획서에 빠짐없이 반영됐는지.
  충족=요구 항목이 모두 반영됨 / 불가=요구 항목이 명백히 누락됨 / 검토=대조자료 미첨부로 확인 불가하거나 필수 여부가 모호함
- 정확성(ACC): 계획서 내용의 사실·규격·수치가 정확하고 RFP와 모순 없는지.
  충족=수치·규격이 정확히 일치 / 불가=명백히 부정확·모순 / 검토=값은 있으나 대조 기준값 자체가 없어 확인 불가
- 검증가능성(VER): 목표·계획이 측정·확인 가능하게(정량·구체) 기술됐는지.
  충족=검증 기준·방법·정량 목표가 명확 / 불가="적절히","최선을 다해" 등 모호 표현뿐 / 검토=정량·모호 표현이 혼재
- 추적성(TRACE): 요구→과업→Task→일정→산출물이 서로 연결·추적 가능하게 기술됐는지.
  충족=단위업무-Task-일정-산출물이 1:1 연결 / 불가=연결이 단절됨 / 검토=일부만 연결되거나 확인 대상 섹션이 입력에 없음

[종합판정 규칙]
- 적용 기준(target_criteria) 중 불가가 하나라도 있으면 → 종합판정은 "불가"
- 불가는 없고 검토가 하나라도 있으면 → 종합판정은 "검토"
- 전부 충족이면 → 종합판정은 "충족"
"""

JUDGE_PROMPT = f"""당신은 과업수행계획서(PEP) 평가 판정을 독립적으로 내리는 평가자입니다.
아래 [평가기준 정의]와 [종합판정 규칙]만 근거로, 주어진 target_criteria에 대해 rfp_excerpt와 pep_excerpt를 직접 대조하여 판정하십시오.
다른 사람이 이미 내린 판정은 제공되지 않습니다 — 당신 스스로 처음부터 판단해야 합니다.

{JUDGE_DEFINITIONS}

target_description.quality에 있는 기준 중 target_criteria로 지정된 것만 판단하십시오.
target_criteria가 2개 이상이면 기준별로 각각 충족/불가/검토를 판단한 뒤, 종합판정 규칙에 따라 overall_verdict를 결정하십시오.

반드시 아래 JSON 형식으로만 답하십시오.
{{
  "criteria_verdicts": {{"완전성": "충족|불가|검토", "정확성": "충족|불가|검토", ...}},
  "overall_verdict": "충족" 또는 "불가" 또는 "검토",
  "reasoning": "기준별 판단 근거를 한두 문장씩"
}}
"""

def judge_item(item, retries=3):
    desc = item['input']['description']
    payload = {
        "target_description": {
            "id": desc.get("id"),
            "section_title": desc.get("section_title"),
            "standard_structure": desc.get("standard_structure"),
        },
        "target_criteria": desc.get("quality", []),
        "rfp_excerpt": item['input'].get('rfp_excerpt', ''),
        "pep_excerpt": item['input'].get('pep_excerpt', ''),
    }
    user_msg = json.dumps(payload, ensure_ascii=False)

    for attempt in range(retries):
        try:
            resp = gclient.models.generate_content(
                model=JUDGE_MODEL,
                contents=f"{JUDGE_PROMPT}\n\n판정 대상:\n{user_msg}",
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json",
                ),
            )
            return json.loads(resp.text)
        except Exception as e:
            print(f"  [warn] judge 호출 실패(attempt {attempt+1}): {e}")
            time.sleep(5)
    return {"overall_verdict": "ERROR", "criteria_verdicts": {}, "reasoning": "judge 호출 실패"}

In [5]:
results = []
mismatches = []

for i, item in enumerate(dataset):
    original_verdict = item['output']['판정']
    gemini = judge_item(item)
    predicted_verdict = gemini.get('overall_verdict')

    agree = (predicted_verdict == original_verdict)
    results.append({
        "id": item.get("id"),
        "gpt_판정": original_verdict,
        "gemini_판정": predicted_verdict,
        "agree": agree,
        "criteria_verdicts": gemini.get("criteria_verdicts"),
        "reasoning": gemini.get("reasoning"),
    })

    tag = "일치" if agree else "불일치"
    print(f"[{i+1}/{len(dataset)}] {item.get('id')} | GPT={original_verdict} vs Gemini={predicted_verdict} -> {tag}")

    if not agree:
        mismatches.append(item.get("id"))

    time.sleep(7)

    with open('agreement_results_pep.json', 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

agreement_rate = sum(r['agree'] for r in results) / len(results)
print(f"\n총 {len(dataset)}개 중 일치 {sum(r['agree'] for r in results)}개 ({agreement_rate:.1%})")
print("불일치 id 목록:", mismatches)

[1/150] PEP-SEED-01 | GPT=충족 vs Gemini=충족 -> 일치
[2/150] PEP-SEED-02 | GPT=불가 vs Gemini=불가 -> 일치
[3/150] PEP-SEED-03 | GPT=충족 vs Gemini=충족 -> 일치
[4/150] PEP-SEED-04 | GPT=불가 vs Gemini=불가 -> 일치
[5/150] PEP-SEED-05 | GPT=불가 vs Gemini=불가 -> 일치
[6/150] PEP-SEED-06 | GPT=검토 vs Gemini=충족 -> 불일치
[7/150] PEP-SEED-07 | GPT=불가 vs Gemini=불가 -> 일치
[8/150] PEP-SEED-08 | GPT=검토 vs Gemini=불가 -> 불일치
[9/150] PEP-SEED-09 | GPT=충족 vs Gemini=충족 -> 일치
[10/150] PEP-SEED-10 | GPT=불가 vs Gemini=불가 -> 일치
[11/150] PEP-SEED-11 | GPT=충족 vs Gemini=충족 -> 일치
[12/150] PEP-SEED-12 | GPT=불가 vs Gemini=불가 -> 일치
[13/150] PEP-SEED-13 | GPT=불가 vs Gemini=불가 -> 일치
[14/150] PEP-SEED-14 | GPT=검토 vs Gemini=불가 -> 불일치
[15/150] PEP-SEED-15 | GPT=충족 vs Gemini=충족 -> 일치
[16/150] PEP-SEED-16 | GPT=불가 vs Gemini=불가 -> 일치
[17/150] PEP-SEED-17 | GPT=충족 vs Gemini=충족 -> 일치
[18/150] PEP-SEED-18 | GPT=불가 vs Gemini=불가 -> 일치
[19/150] PEP-SEED-19 | GPT=검토 vs Gemini=불가 -> 불일치
[20/150] PEP-SEED-20 | GPT=충족 vs Gemini=충족 -> 일치
[21/150] PEP-SEED-21 | GP

In [6]:
from collections import Counter
confusion = Counter((r['gpt_판정'], r['gemini_판정']) for r in results)
print("혼동행렬 (GPT라벨, Gemini라벨) -> 개수")
for k, v in sorted(confusion.items(), key=lambda x: -x[1]):
    marker = "  <- 불일치" if k[0] != k[1] else ""
    print(f"  {k}: {v}{marker}")

혼동행렬 (GPT라벨, Gemini라벨) -> 개수
  ('불가', '불가'): 62
  ('검토', '불가'): 41  <- 불일치
  ('충족', '충족'): 35
  ('충족', '불가'): 6  <- 불일치
  ('검토', '충족'): 4  <- 불일치
  ('충족', '검토'): 2  <- 불일치


### 불일치 항목 상세 분석

In [7]:
for r in results:
    if not r['agree']:
        print(f"\nID: {r['id']}")
        print(f"  GPT 판정: {r['gpt_판정']}")
        print(f"  Gemini 판정: {r['gemini_판정']}")
        print(f"  기준별 판정: {r['criteria_verdicts']}")
        print(f"  Gemini 추론: {r['reasoning']}")


ID: PEP-SEED-06
  GPT 판정: 검토
  Gemini 판정: 충족
  기준별 판정: {'완전성': '충족', '정확성': '충족'}
  Gemini 추론: 완전성 측면에서 RFP가 요구한 43개 항목 반영 및 데이터 연계 방안이 계획서에 모두 포함되었습니다. 정확성 측면에서 RFP의 요구사항 총괄표 준수 및 연계 방식(REST API) 등 기술적 내용이 모순 없이 일치합니다.

ID: PEP-SEED-08
  GPT 판정: 검토
  Gemini 판정: 불가
  기준별 판정: {'완전성': '불가'}
  Gemini 추론: RFP에서 요구한 외부 안전진단 전문기관 자문위원 포함 여부와 사업자-발주기관·이용기관 간의 협업 절차에 대한 내용이 계획서에 전혀 기술되지 않아 요구사항이 누락되었습니다.

ID: PEP-SEED-14
  GPT 판정: 검토
  Gemini 판정: 불가
  기준별 판정: {'완전성': '불가', '추적성': '불가'}
  Gemini 추론: 완전성 측면에서 RFP가 요구한 213개교 시스템 구축 관련 필수 산출물 및 제출부수 정보가 누락되었으며, 추적성 측면에서 일부 산출물의 제출일정이 명시되지 않았고 전체적인 과업-산출물-일정 간의 연결 체계가 불완전합니다.

ID: PEP-SEED-19
  GPT 판정: 검토
  Gemini 판정: 불가
  기준별 판정: {'완전성': '불가', '검증가능성': '불가'}
  Gemini 추론: 완전성 측면에서 RFP가 요구한 공공데이터 표준화 지침 및 웹접근성 준수 확인 항목이 일부 누락되었으며, 검증가능성 측면에서 웹품질 개선 계획이 '최선을 다해'와 같은 모호한 표현으로 기술되어 정량적 검증이 불가능함.

ID: PEP-SEED-24
  GPT 판정: 검토
  Gemini 판정: 불가
  기준별 판정: {'완전성': '불가', '검증가능성': '검토'}
  Gemini 추론: 완전성 측면에서 RFP가 요구한 '외부 의료정보 보안 전문가 자문 반영' 내용이 계획서에 누락되었습니다. 검